In [60]:
import pandas as pd
import itertools
import os
import random
from tqdm import tqdm
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from openai import OpenAI

In [61]:
from dotenv import load_dotenv

load_dotenv()

True

In [62]:
level_quality = 2  # Threshold for good/bad MCQs
#data_dir = "raiforukraine-MARIA/data/data_for_finetuning/Training_mcqs/"
#datasets = ['gemma_9b_distractor_quality', 'Llama8B_0.1_distractor_quality', 'Llama1b_distractor_quality']  # Add 'phi3.5' when available
datasets = ['llama3_1_8b','openbiollm_8b', "gemma2_9b", "medGemma_4b", "medGemma_27b"]  # Add 'phi3.5' when available
data_dir = "../data/dataset_with_quality/"
OPENAI_KEY = os.getenv('OPENAI_KEY')
max_gpt4o_attempts = 3

In [63]:
client = OpenAI(api_key=OPENAI_KEY)

In [64]:
class MCQQuestion(BaseModel):
    question: str = Field(description="The multiple-choice question")
    option_a: str = Field(description="The first answer option labeled 'A'")
    option_b: str = Field(description="The second answer option labeled 'B'")
    option_c: str = Field(description="The third answer option labeled 'C'")
    option_d: str = Field(description="The fourth answer option labeled 'D'")
    correct_option: str = Field(description="This consists only a letter of the correct option")

mcq_parser = JsonOutputParser(pydantic_object=MCQQuestion)

In [65]:
DEFAULT_PROMPT = """You are an expert in creating high-quality medical multiple-choice questions.
Create a challenging medical multiple-choice question with exactly one correct answer and three plausible distractors.

The distractors (incorrect options) quality is extremely important and should meet these criteria:
- They should be plausible to most test-takers
- They should represent common misconceptions
- They should require deep understanding to eliminate
- They should not be obviously incorrect or unrelated to the question
- They should not be easy to eliminate for knowledgeable test-takers

The question should test medical knowledge and critical thinking skills.
"""

In [66]:
DISTRACTORS_QUALITY_PROMPT = """You are tasked to evaluate the quality of the distractors of a multiple-choice question (incorrect options) on a scale of 1-5, where:
1 = POOR: Implausible, obviously incorrect, or unrelated to the question
2 = BELOW AVERAGE: Easy to eliminate, lacks plausibility for knowledgeable test-takers  
3 = AVERAGE: Somewhat plausible but contains minor flaws that make it distinguishable
4 = GOOD: Plausible to most test-takers, represents common misconceptions
5 = EXCELLENT: Highly plausible, represents sophisticated misconceptions, requires deep understanding to eliminate
Provide only a numerical score from 1 to 5 the best represents the level of distractor quality"""


In [67]:
def create_prompt_chain(prompt_text=DEFAULT_PROMPT):
    prompt_template = PromptTemplate(
        template="{prompt}.\n{format_instructions}\n{query}\n",
        input_variables=["query"],
        partial_variables={
            "prompt": prompt_text,
            "format_instructions": mcq_parser.get_format_instructions(),
        },
    )
    model = ChatOpenAI(model="gpt-4o", temperature=0.5, api_key=OPENAI_KEY)
    
    chain = prompt_template | model | mcq_parser
    return chain

In [68]:
def call_openai_api(client, system_prompt, user_prompt, temp=0.5, max_completion_tokens=1):
    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            temperature=temp,
            max_tokens=max_completion_tokens,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"Error occurred: {e}")
        return None

In [69]:
def generate_prompt_for_question(row,
                                question_col='question',
                                option_a_col='option_a',
                                option_b_col='option_b',
                                option_c_col='option_c',
                                option_d_col='option_d',
                                correct_option='correct_option',
                                include_options=True,
                                include_correct_option=True,
                                context_col=None):
    question_text = row[question_col]
    options = f"a) {row[option_a_col]}\nb) {row[option_b_col]}\nc) {row[option_c_col]}\nd) {row[option_d_col]}"
    correct_opt = row[correct_option]
    
    user_prompt_delimiter = "-----\n"
    user_prompt_question = f"Question:\n{question_text}\n"
    user_prompt_options = f"Options:\n{options}\n"
    user_prompt = user_prompt_delimiter + user_prompt_question
    if include_options:
        user_prompt += user_prompt_options
    if include_correct_option:
        correct_option_text = f"Correct option: {correct_opt}\n"
        user_prompt += correct_option_text
    user_prompt += user_prompt_delimiter
    
    if context_col is not None and context_col in row:
        mcq_context = f"""Context:\n-----\n{row[context_col]}\n-----\n"""
        user_prompt = mcq_context + user_prompt
    return user_prompt

In [70]:
def evaluate_distractor_quality(mcq_dict):
    # Convert dictionary to a pandas Series for use with generate_prompt_for_question
    row = pd.Series(mcq_dict)
    
    # Generate prompt for evaluation
    user_prompt = generate_prompt_for_question(
        row,
        question_col='question',
        option_a_col='option_a',
        option_b_col='option_b',
        option_c_col='option_c',
        option_d_col='option_d',
        correct_option='correct_option',
        include_options=True,
        include_correct_option=True
    )
    
    # Call API to evaluate
    quality_score = call_openai_api(
        client,
        DISTRACTORS_QUALITY_PROMPT,
        user_prompt,
        temp=0.5,
        max_completion_tokens=1
    )
    
    try:
        return int(quality_score.strip())
    except (ValueError, TypeError):
        print(f"Error parsing quality score: {quality_score}")
        return 0

In [71]:
def generate_good_mcq_with_gpt4o(question_text):
    try:
        # Create the chain
        chain = create_prompt_chain()
        
        # Generate MCQ using the DEFAULT_PROMPT
        # Just pass the question topic without additional instructions
        query = question_text
        generated_mcq = chain.invoke({"query": query})
        
        # Evaluate distractor quality
        quality = evaluate_distractor_quality(generated_mcq)
        
        return generated_mcq, quality
        
    except Exception as e:
        print(f"Error generating MCQ with GPT-4o: {e}")
        return None, 0

In [72]:
def create_mcq_text(mcq_dict):
    return (
        f"Question: {mcq_dict['question']}\n"
        f"a) {mcq_dict['option_a']}\n"
        f"b) {mcq_dict['option_b']}\n"
        f"c) {mcq_dict['option_c']}\n"
        f"d) {mcq_dict['option_d']}"
    )

# Function to create a pair dictionary
def create_pair(lisa_id, good_mcq, bad_mcq, good_source="", bad_source=""):
    good_text = create_mcq_text(good_mcq)
    bad_text = create_mcq_text(bad_mcq)
    
    return {
        "id": lisa_id,
        "chosen": good_text,
        "rejected": bad_text,
        "chosen_source": good_source,
        "rejected_source": bad_source
    }

In [73]:
print("Processing MCQ datasets...")

id2mcqs = dict()  # Key: dataset, Value: {id: mcq_dict}
ids = set()       # Set of all common IDs across datasets
good_mcqs = dict() # Key: id, Value: list of datasets with good MCQs
bad_mcqs = dict()  # Key: id, Value: list of datasets with bad MCQs

Processing MCQ datasets...


In [74]:
quality_null = 0
df_null = pd.DataFrame()
tot = 0
for dataset in datasets:
    try:
        file_path = os.path.join(data_dir, f"{dataset}.csv")
        df = pd.read_csv(file_path)
        df_null = pd.concat([ df_null, df[df["distractor_quality"] == 0]] , ignore_index=True)
        tot += len(df)
        # df = df.iloc[:50]
        
        # Extract MCQ content and ensure correct_option is included
        temp = df[['id', 'question', 'option_a', 'option_b', 'option_c', 'option_d', 'correct_option']].set_index('id').T.to_dict()
        
        # Update set of common IDs
        if len(ids) == 0:
            ids = set(temp.keys())
        else:
            ids = ids.intersection(set(temp.keys()))
        
        # Store MCQs by dataset
        id2mcqs[dataset] = temp.copy()
        
        # Categorize MCQs by quality
        for idx, row in df[['id', 'distractor_quality']].iterrows():
            lisa_id, quality = row
            #if quality == 0:
            #    quality_null += 1
            #    continue
            
            # Good MCQ
            if quality > level_quality:
                if lisa_id in good_mcqs:
                    good_mcqs[lisa_id].append(dataset)
                else:
                    good_mcqs[lisa_id] = [dataset]
            else:
                if lisa_id in bad_mcqs:
                    bad_mcqs[lisa_id].append(dataset)
                else:
                    bad_mcqs[lisa_id] = [dataset]
                    
    except Exception as e:
        print(f"Error processing dataset {dataset}: {e}")

print(f"Found {len(ids)} common IDs across all datasets")
print(f"Good MCQs found for {len(good_mcqs)} IDs")
print(f"Bad MCQs found for {len(bad_mcqs)} IDs")
print(f"Invalid MCQs correct answer not corresponding to the question : {quality_null}")

Found 1592 common IDs across all datasets
Good MCQs found for 1569 IDs
Bad MCQs found for 1361 IDs
Invalid MCQs correct answer not corresponding to the question : 0


In [55]:
print(tot)

7960


In [56]:
def create_mcq_text_(mcq_dict):
    return (
        f"Question: {mcq_dict['question']}\n"
        f"a) {mcq_dict['option_a']}\n"
        f"b) {mcq_dict['option_b']}\n"
        f"c) {mcq_dict['option_c']}\n"
        f"d) {mcq_dict['option_d']}\n\n"
        f"R : {mcq_dict['correct_option']}"
    )
df_null.to_csv("quality_null.csv", index=False)
lisa_sheets = pd.read_csv("../data/lisa_sheets.csv")
for idx, row in df_null.iterrows():
    print(create_mcq_text_(row))
    print()
    print(lisa_sheets[ lisa_sheets ["id"] == row["id"]] ["content_raw"] .iloc[0]  )


len(df_null)

Question: Quels sont les professionnels qui engagent leur responsabilité civile personnelle en cas de dommage causé à un patient ?
a) Les médecins salariés d’établissements, services ou organismes privés
b) Les praticiens hospitaliers au sein d’un établissement public de santé
c) Tous les professionnels de la santé, quels que soient leur statut et leur lieu d’exercice
d) Seuls les médecins libéraux

R : d

{{objectif de connaissance
|Identifiant=OIC-005-03-A
|Item_parent=Responsabilités médicale pénale, civile, administrative et disciplinaire. La gestion des erreurs et des plaintes ; l’aléa thérapeutique
|Item_parent_short=Responsabilités médicale pénale, civile, administrative et disciplinaire. La gestion des erreurs et des plaintes ; l’aléa thérapeutique
|Rang=A
|Intitulé=Connaître la définition de la responsabilité civile
|Description=None
|Rubrique=Définition
|Contributeurs=Bastien Boussat
|Ordre=3}}

Les professionnels exerçant à titre libéral et les établissements de santé privés

2627

In [75]:
for dataset in datasets:
    file_path = os.path.join(data_dir, f"{dataset}.csv")
    df = pd.read_csv(file_path)
    print(f"{dataset} quality distribution:")
    print(df['distractor_quality'].value_counts())


llama3_1_8b quality distribution:
distractor_quality
5    993
0    488
4    108
3      2
1      1
Name: count, dtype: int64
openbiollm_8b quality distribution:
distractor_quality
0    1066
5     420
4     100
3       5
1       1
Name: count, dtype: int64
gemma2_9b quality distribution:
distractor_quality
5    1117
0     271
4     199
3       4
1       1
Name: count, dtype: int64
medGemma_4b quality distribution:
distractor_quality
5    905
0    558
4    124
1      4
3      1
Name: count, dtype: int64
medGemma_27b quality distribution:
distractor_quality
5    1259
0     244
4      88
1       1
Name: count, dtype: int64


In [76]:
nb_good_only = 0
nb_bad_only = 0
nb_mixed = 0
nb_gpt4o_generated = 0
all_pairs = []

# Process each ID to create pairs
for idx in tqdm(ids, desc="Creating pairs"):
    bad_mcqs_list = []
    good_mcqs_list = []
    
    if idx in bad_mcqs:
        bad_datasets = bad_mcqs[idx]
        for dataset in bad_datasets:
            if idx in id2mcqs[dataset]:
                bad_mcqs_list.append((id2mcqs[dataset][idx], dataset))
    else:
        # All MCQs are good
        nb_good_only += 1
        continue
        
    if idx in good_mcqs:
        good_datasets = good_mcqs[idx]
        for dataset in good_datasets:
            if idx in id2mcqs[dataset]:
                good_mcqs_list.append((id2mcqs[dataset][idx], dataset))
        nb_mixed += 1
    else:
        # All MCQs are bad - try to generate a good one with GPT-4o
        nb_bad_only += 1
        
        # Get sample question from one of the bad MCQs
        sample_question = bad_mcqs_list[0][0]["question"]
        
        # Try to generate a good MCQ with GPT-4o (up to max_attempts times)
        for attempt in range(max_gpt4o_attempts):
            #generated_mcq, quality = generate_good_mcq_with_gpt4o(sample_question)
            generated_mcq, quality = {"question" : "", "option_a" : "", "option_b" : "", "option_c" : "", "option_d": "", "correct_option" : ""}, 4
            
            if generated_mcq and quality >= level_quality:  # Aim for quality 3 or higher
                good_mcqs_list.append((generated_mcq, "gpt-4o"))
                nb_gpt4o_generated += 1
                print(f"Successfully generated MCQ with quality score {quality} for ID {idx}")
                break
            else:
                print(f"Attempt {attempt+1}: Generated MCQ quality ({quality}) below threshold of {level_quality} for ID {idx}")
        
        # If we couldn't generate a good MCQ, skip this ID
        if not good_mcqs_list:
            print(f"Could not generate good MCQ for ID {idx} after {max_gpt4o_attempts} attempts. Skipping.")
            continue
    
    # Create all possible (good, bad) pairs
    for good_item, bad_item in itertools.product(good_mcqs_list, bad_mcqs_list):
        good_mcq, good_source = good_item
        bad_mcq, bad_source = bad_item
        
        pair = create_pair(idx, good_mcq, bad_mcq, good_source, bad_source)
        all_pairs.append(pair)

# Convert to DataFrame
pairs_df = pd.DataFrame(all_pairs)
# Check how many IDs have all good or all bad MCQs
print(f"IDs with all good MCQs: {nb_good_only}")
print(f"IDs with all bad MCQs: {nb_bad_only}")
print(f"IDs with mixed quality MCQs: {len(ids) - nb_good_only - nb_bad_only}")
print("length of all pairs:", len(all_pairs))

Creating pairs: 100%|██████████| 1592/1592 [00:00<00:00, 7882.50it/s]

Successfully generated MCQ with quality score 4 for ID OIC-157-12-A
Successfully generated MCQ with quality score 4 for ID OIC-101-09-B
Successfully generated MCQ with quality score 4 for ID OIC-342-03-A
Successfully generated MCQ with quality score 4 for ID OIC-019-04-B
Successfully generated MCQ with quality score 4 for ID OIC-047-02-B
Successfully generated MCQ with quality score 4 for ID OIC-235-14-B
Successfully generated MCQ with quality score 4 for ID OIC-284-03-B
Successfully generated MCQ with quality score 4 for ID OIC-263-07-B
Successfully generated MCQ with quality score 4 for ID OIC-101-04-A
Successfully generated MCQ with quality score 4 for ID OIC-032-08-B
Successfully generated MCQ with quality score 4 for ID OIC-197-09-A
Successfully generated MCQ with quality score 4 for ID OIC-162-02-B
Successfully generated MCQ with quality score 4 for ID OIC-037-05-A
Successfully generated MCQ with quality score 4 for ID OIC-038-09-B
Successfully generated MCQ with quality score 4 

In [59]:
print(pairs_df)

              id                                             chosen  \
0   OIC-302-13-A  Question: Quel est l'objectif principal de la ...   
1   OIC-302-13-A  Question: Quelle est l'approche principale de ...   
2   OIC-302-13-A  Question: Quelle est la principale stratégie d...   
3   OIC-101-09-B                     Question: \na) \nb) \nc) \nd)    
4   OIC-010-01-A  Question: Selon le texte, comment les représen...   
5   OIC-153-04-A  Question: Quelle est la principale cause des e...   
6   OIC-153-04-A  Question: Quel est le principal risque associé...   
7   OIC-153-04-A  Question: Quel est le principal risque infecti...   
8   OIC-153-04-A  Question: Quel est le principal risque infecti...   
9   OIC-020-10-B  Question: Qu'est-ce qu'une variable aléatoire ...   
10  OIC-247-05-B  Question: Quel est le diabète qui se définit p...   
11  OIC-247-05-B  Question: Quel est le critère principal pour d...   
12  OIC-247-05-B  Question: Lequel des énoncés suivants décrit l...   
13  OI

In [77]:
# Statistics
print(f"Statistics:")
print(f"  IDs with all good MCQs: {nb_good_only}")
print(f"  IDs with all bad MCQs: {nb_bad_only}")
print(f"  IDs with mixed quality MCQs: {nb_mixed}")
print(f"  MCQs generated with GPT-4o: {nb_gpt4o_generated}")
print(f"  Total pairs created: {len(pairs_df)}")

# Split into training and evaluation datasets
eval_size = min(len(pairs_df) // 10, 200)  # 10% or max 200 samples for evaluation
eval_pairs = pairs_df.sample(n=eval_size, random_state=42)
train_pairs = pairs_df.drop(eval_pairs.index)

# Save to CSV
pairs_df.to_csv("../data/finetuning_datasets/all_mcq_pairs.csv", index=False)
train_pairs.to_csv("../data/finetuning_datasets/train_mcq_pairs_for_dpo.csv", index=False)
eval_pairs.to_csv("../data/finetuning_datasets/eval_mcq_pairs_for_dpo.csv", index=False)

print(f"Created {len(train_pairs)} training pairs and {len(eval_pairs)} evaluation pairs")
print("Files saved as all_mcq_pairs.csv, train_mcq_pairs_for_dpo.csv, and eval_mcq_pairs_for_dpo.csv")

Statistics:
  IDs with all good MCQs: 231
  IDs with all bad MCQs: 23
  IDs with mixed quality MCQs: 1338
  MCQs generated with GPT-4o: 23
  Total pairs created: 6861
Created 6661 training pairs and 200 evaluation pairs
Files saved as all_mcq_pairs.csv, train_mcq_pairs_for_dpo.csv, and eval_mcq_pairs_for_dpo.csv


In [217]:
print(f'{nb_good_only} of Lisa sheets ({nb_good_only/len(ids)*100}%) gives only good MCQs across all LLMs')
print(f'{nb_bad_only} of Lisa sheets ({nb_bad_only/len(ids)*100}%) gives only bad MCQs across all LLMs')

231 of Lisa sheets (14.510050251256281%) gives only good MCQs across all LLMs
23 of Lisa sheets (1.4447236180904524%) gives only bad MCQs across all LLMs
